# Exercício 03 — EduPrompt Academy

**Versão didática:** todo o código LangChain/LCEL está **neste notebook** (sem importar `app.chains`). Assim consegues ler de cima a baixo: templates → chains → paralelo → exemplo.

> A pasta `app/` mantém uma cópia modular para **Docker/FastAPI** opcional; o enunciado «sem ecrã» cumpre-se aqui no Jupyter.

**Empresa simulada:** a EduPrompt Academy vende trilhas educacionais com IA. **Problema:** professores precisam de **explicações**, **exercícios** e **resumos** em pouco tempo.

**Frameworks:** LangChain (`langchain_core`), **LCEL** (`|` e `RunnableParallel`), Gemini via `langchain-google-genai`, Docker (ver `README.md`).

## Conceitos antes do código

### Arquitetura pedida pelo enunciado

```text
Tema informado (tema, nivel)
  → PromptTemplate (aqui: ChatPromptTemplate para modelo de chat)
  → Modelo LLM
  → Parser
  → Resposta educacional
```

### Por que `ChatPromptTemplate` e não só «PromptTemplate»?

Modelos **de chat** (Gemini, GPT-4, etc.) consomem **mensagens** com papéis (`system`, `human`, `ai`). O objeto adequado no LangChain é o **`ChatPromptTemplate`**: defines o «guião» dessas mensagens com variáveis `{tema}` e `{nivel}`. O `PromptTemplate` clássico é mais usado quando se envia **uma única string** a um modelo tipo *completion*.

### LCEL — LangChain Expression Language

- O operador **`|`** liga **Runnable** em sequência: saída de um passo é entrada do seguinte.
- Exemplo mental: `A | B | C` é «primeiro corre `A`, depois `B`, depois `C`».
- **`StrOutputParser()`** no fim converte a resposta do chat numa **`str`** Python simples — ideal para juntar Markdown ou gravar ficheiros.

### `RunnableParallel`

Quando várias chains precisam **das mesmas variáveis** (`tema`, `nivel`) mas produzem **saídas diferentes**, podes agrupá-las: um único `.invoke({...})` devolve um dicionário com uma chave por chain (ex.: `explicacao`, `exercicios`, `resumo`). Isto é composição LCEL — não é obrigatório que o modelo execute fisicamente em paralelo; é uma **abstração** limpa para fluxos paralelos ou independentes.

## 0. Ambiente e caminhos

Carregamos `.env` na raiz do repositório do curso (onde deve estar `GOOGLE_API_KEY` para o Gemini).

In [ ]:
from pathlib import Path
import os

from dotenv import load_dotenv

EX_ROOT = Path.cwd().resolve()
REPO_ROOT = EX_ROOT.parent.parent

load_dotenv(REPO_ROOT / ".env", override=False)
load_dotenv(EX_ROOT / ".env", override=True)

if not (os.environ.get("GOOGLE_API_KEY") or os.environ.get("GEMINI_API_KEY")):
    raise RuntimeError("Defina GOOGLE_API_KEY no .env na raiz do repositório.")

print("OK — Repo:", REPO_ROOT)
print("OK — Exercício:", EX_ROOT)

## 1. Imports LangChain e instância do modelo

Importamos **apenas bibliotecas externas** — nada da pasta `app` do projeto.

- **`ChatGoogleGenerativeAI`**: cliente do Gemini compatível com LCEL.
- **`temperature` baixa** (0,25): respostas mais estáveis para material pedagógico.

In [ ]:
import json

from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableParallel
from langchain_google_genai import ChatGoogleGenerativeAI

_model_name = (os.environ.get("GEMINI_MODEL") or "gemini-2.0-flash").replace("models/", "")

llm = ChatGoogleGenerativeAI(model=_model_name, temperature=0.25)
print("Modelo:", _model_name)

## 2. Três templates de prompt (`ChatPromptTemplate`)

Cada template declara **variáveis** `tema` e `nivel`. No `.invoke({"tema": "RAG", "nivel": "iniciante"})` esses espaços são preenchidos.

| Chain | Função pedagógica |
|-------|-------------------|
| **Explicação** | Texto contínuo, 2–4 parágrafos; sem `##` no corpo (o notebook acrescenta cabeçalhos depois). |
| **Exercícios** | Exatamente **três** itens numerados `1.` … `3.` |
| **Resumo** | 4–6 bullets com `- ` |

A mensagem **`system`** fixa o papel (EduPrompt Academy, português europeu). A **`human`** traz as instruções específicas por tarefa.

In [ ]:
PROMPT_EXPLICACAO = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "És o EduPrompt Academy. Escreve em português europeu, tom professoral e claro. "
            "Adaptas vocabulário e profundidade ao nível pedido (iniciante, intermédio ou avançado).",
        ),
        (
            "human",
            "Explica o tema «{tema}» para um público de nível «{nivel}».\n"
            "Usa 2–4 parágrafos curtos. Começa por contextualizar o tema numa frase.\n"
            "Não uses cabeçalhos Markdown (##) no texto; só parágrafos e listas curtas se ajudarem.",
        ),
    ]
)

PROMPT_EXERCICIOS = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "És o EduPrompt Academy. Crias exercícios objetivos em português europeu.",
        ),
        (
            "human",
            "Cria **três** exercícios sobre «{tema}», adequados ao nível «{nivel}».\n"
            "Numera exatamente assim: `1.`, `2.`, `3.` — uma linha ou parágrafo curto por exercício.\n"
            "Sem cabeçalhos Markdown.",
        ),
    ]
)

PROMPT_RESUMO = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "És o EduPrompt Academy. Resumes com rigor em português europeu.",
        ),
        (
            "human",
            "Resume «{tema}» para nível «{nivel}» em 4 a 6 bullet points.\n"
            "Cada linha começa por `- `. Sem cabeçalhos Markdown.",
        ),
    ]
)

## 3. Três chains LCEL: `template | llm | StrOutputParser()`

O operador **`|`** constrói um **`RunnableSequence`**:

1. O template transforma `{"tema", "nivel"}` numa lista de mensagens para o modelo.
2. O `llm` gera a resposta.
3. O **`StrOutputParser`** extrai só o texto.

Cada função abaixo devolve uma **chain reutilizável** — podes guardá-la numa variável e chamar `.invoke(...)` várias vezes com temas diferentes.

In [ ]:
def chain_explicacao():
    return PROMPT_EXPLICACAO | llm | StrOutputParser()


def chain_exercicios():
    return PROMPT_EXERCICIOS | llm | StrOutputParser()


def chain_resumo():
    return PROMPT_RESUMO | llm | StrOutputParser()

## 4. Modo sequencial (três chamadas ao modelo)

Útil para perceber **custo/latência**: cada `.invoke` é uma ida ao Gemini. Comparando com a secção seguinte, vês quando faz sentido agrupar logicamente em paralelo.

In [ ]:
entrada = {"tema": "RAG", "nivel": "iniciante"}
print("Entrada:", json.dumps(entrada, ensure_ascii=False, indent=2))

texto_exp = chain_explicacao().invoke(entrada)
texto_exe = chain_exercicios().invoke(entrada)
texto_res = chain_resumo().invoke(entrada)

print("\n[Sequencial] primeiros 300 chars da explicação:\n", texto_exp[:300], "…")

## 5. `RunnableParallel` — uma invocação, três saídas

`RunnableParallel` recebe **chains nomeadas**. O mesmo dicionário `entrada` é passado a cada uma. O resultado é um **dict** Python:

```python
{"explicacao": "...", "exercicios": "...", "resumo": "..."}
```

Depois juntamos as partes no formato do enunciado: cabeçalhos `## Explicação`, `## Exercícios`, `## Resumo`.

In [ ]:
def parallel_educacional():
    return RunnableParallel(
        explicacao=chain_explicacao(),
        exercicios=chain_exercicios(),
        resumo=chain_resumo(),
    )


def montar_markdown_pacote(partes: dict) -> str:
    return (
        "## Explicação\n\n"
        f"{partes['explicacao'].strip()}\n\n"
        "## Exercícios\n\n"
        f"{partes['exercicios'].strip()}\n\n"
        "## Resumo\n\n"
        f"{partes['resumo'].strip()}"
    )


def gerar_pacote_educacional(vars_: dict, *, paralelo: bool = True) -> dict:
    if paralelo:
        out = parallel_educacional().invoke(vars_)
    else:
        out = {
            "explicacao": chain_explicacao().invoke(vars_),
            "exercicios": chain_exercicios().invoke(vars_),
            "resumo": chain_resumo().invoke(vars_),
        }
    return {**out, "markdown": montar_markdown_pacote(out)}


par_out = parallel_educacional().invoke(entrada)
print("Chaves devolvidas pelo RunnableParallel:", list(par_out.keys()))

pacote = gerar_pacote_educacional(entrada, paralelo=True)
print("\n--- Markdown consolidado (exemplo de saída pedido no enunciado) ---\n")
print(pacote["markdown"])

## 6. Desafio extra — quarta chain (narrativa sarcástica nerd)

Mesmo padrão LCEL: novo template + mesmo `llm` + `StrOutputParser`. Tom **irónico leve**, sem ataques a pessoas reais — só caricatura de dinâmicas Produto vs Engenharia.

In [ ]:
PROMPT_NARRATIVA_NERD = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "Humor nerd e sarcástico **leve**; não insultes grupos nem pessoas. Mantém correção técnica.",
        ),
        (
            "human",
            "Escreve **um único parágrafo** (até ~120 palavras) sobre «{tema}», "
            "como se fossem duas equipas (Produto vs Engenharia) a «lutar» pelo mesmo buzzword, "
            "para um leitor «{nivel}». Tom ironico corporativo de TI.",
        ),
    ]
)


def chain_narrativa_nerd():
    return PROMPT_NARRATIVA_NERD | llm | StrOutputParser()


print(chain_narrativa_nerd().invoke(entrada))

## Referências fora do notebook

- [`docs/chains.md`](docs/chains.md) — descrição textual de cada chain (espelha o que implementaste acima).
- [`docs/explicacao_teorica.md`](docs/explicacao_teorica.md), [`docs/arquitetura.md`](docs/arquitetura.md).
- [`app/chains/eduprompt_chains.py`](app/chains/eduprompt_chains.py) — **mesma lógica** packaging para `app/main.py` / Docker API opcional; não é necessário para correr este notebook.